# Sepedi Fine-Tuning — FINAL EVALUATION notebook
Loads the trained model and the held-out test set saved by `training.ipynb`, runs `trainer.evaluate()` **exactly once**.
This is the number that goes in the report -- do not re-run training here, and do not use this test set for any decisions.

## Kaggle Setup

In [ ]:
import importlib
for pkg in ["torch", "transformers", "datasets", "evaluate", "sklearn"]:
    try:
        importlib.import_module(pkg)
        print(f"{pkg}: OK")
    except ImportError:
        print(f"{pkg}: MISSING")
        !pip install -q {pkg}

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import pandas as pd
import numpy as np
import evaluate
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

seed = 100
np.random.seed(seed)
torch.manual_seed(seed)

## Load Model + Held-Out Test Set — EDIT THE PATH BELOW to match your uploaded Kaggle Model
(the folder you get from `training.ipynb`'s last cell, uploaded via Kaggle Models)

In [ ]:
# EDIT: replace with your actual uploaded Sepedi model path
model_path = "/kaggle/input/YOUR-SEPEDI-MODEL-NAME/afroxlmr_sepedi_final"

id2label = {0: "not misinformation", 1: "misinformation"}
label2id = {"not misinformation": 0, "misinformation": 1}

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

test_df = pd.read_csv(f"{model_path}/held_out_test_set.csv", encoding="utf-8")
print(f"Held-out test set: {len(test_df)} rows")
print(test_df["label"].value_counts())

test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

## Tokenize

In [ ]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_test = test_dataset.map(preprocess_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Metrics

In [ ]:
accuracy_metric = evaluate.load("accuracy")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"],
        "recall": recall_metric.compute(predictions=predictions, references=labels)["recall"],
        "precision": precision_metric.compute(predictions=predictions, references=labels)["precision"],
        "f1": f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"],
    }

## Final Evaluation — run exactly once, this is the reportable number

In [ ]:
eval_args = TrainingArguments(
    output_dir="/kaggle/working/final_eval_scratch",
    per_device_eval_batch_size=16,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=eval_args,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

final_results = trainer.evaluate(eval_dataset=tokenized_test)
print("FINAL TEST RESULTS (report this):")
print(final_results)

## Save Results

In [ ]:
import json
with open("/kaggle/working/sepedi_final_test_results.json", "w") as f:
    json.dump(final_results, f, indent=2)
print("Saved to /kaggle/working/sepedi_final_test_results.json")